In [40]:
%matplotlib qt
import warnings, os, time
warnings.simplefilter("ignore")
import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve
from scipy.ndimage import gaussian_filter1d
from astropy.stats import sigma_clip, mad_std
from astroquery.mast import Observations

Observations.TIMEOUT = 600
Observations.PAGESIZE = 5000

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 9

SETORES = [1, 27]
CACHE   = "lc_cache"
os.makedirs(CACHE, exist_ok=True)


def com_retry(func, tentativas=6, espera=10):
    for i in range(tentativas):
        try:
            return func()
        except Exception as e:
            print(f"    tentativa {i+1}/{tentativas} falhou: {type(e).__name__}")
            if i == tentativas - 1:
                raise
            time.sleep(espera * (i + 1))


t_por_setor, f_por_setor, mask_por_setor = {}, {}, {}

for s in SETORES:
    arq = os.path.join(CACHE, f"AUMic_setor{s}.npz")

    if os.path.exists(arq):
        d = np.load(arq)
        t_sec, f_sec = d['t'], d['f']
        print(f"Setor {s}: lido do cache ({arq})")
    else:
        print(f"Setor {s}: consultando MAST...")
        sr = com_retry(lambda: search_lightcurve("AU Mic", mission="TESS", author="SPOC",
                                                 exptime=120, sector=s))
        if len(sr) == 0:
            print(f"[!] Setor {s} nao encontrado."); continue

        lc = com_retry(lambda: sr[0].download())
        lc = lc[lc.quality == 0].remove_nans().normalize()   # SEM remove_outliers()

        t_sec = np.ascontiguousarray(lc.time.value, dtype=np.float64)
        f_sec = np.ascontiguousarray(lc.flux.value, dtype=np.float64)
        ok = np.isfinite(t_sec) & np.isfinite(f_sec)
        t_sec, f_sec = t_sec[ok], f_sec[ok]
        ordem = np.argsort(t_sec)
        t_sec, f_sec = t_sec[ordem], f_sec[ordem]

        np.savez_compressed(arq, t=t_sec, f=f_sec)
        print(f"  salvo em {arq}")

    desvio = f_sec - gaussian_filter1d(f_sec, sigma=15)
    mask_good_flares = desvio < (3 * np.nanstd(desvio))

    t_por_setor[s], f_por_setor[s], mask_por_setor[s] = t_sec, f_sec, mask_good_flares
    print(f"  Setor {s:>2d}: {t_sec.size} pontos | BTJD {t_sec.min():.3f} -> {t_sec.max():.3f} "
          f"| {(~mask_good_flares).sum()} pts marcados como flare")

Setor 1: lido do cache (lc_cache\AUMic_setor1.npz)
  Setor  1: 17687 pontos | BTJD 1325.944 -> 1353.045 | 124 pts marcados como flare
Setor 27: lido do cache (lc_cache\AUMic_setor27.npz)
  Setor 27: 16767 pontos | BTJD 2036.284 -> 2060.647 | 123 pts marcados como flare


In [ ]:
# ============================================================
# AJUSTE DA BASELINE
#   Setor 1  -> ajuste LOCAL (regioes na mao) - o automatico nao funciona aqui
#   Setor 27 -> ajuste POLINOMIAL automatico por efemeride
#   Janela de extracao: (-6,+6) h em torno de Tc para TODOS os transitos
# ============================================================
P_REF   = 8.4631516
T0_REF  = 1330.39051        # Tc do artigo
T0_ERR  = 0.00015           # incerteza em Tc (dias)
DUR     = 3.50 / 24.0       # duracao do transito do artigo (3.50 h)
DUR_ERR = 0.08 / 24.0       # incerteza na duracao (0.08 h)

# --- janela unica de extracao/ajuste: (-6,+6) h de Tc ---
JAN_H   = 6.0
JANELA  = JAN_H / 24.0      # = 0.25 d

# --- parametros do polinomial automatico (Setor 27) ---
GRAU    = 3
FATOR   = 1.3
SIG_UP, SIG_LOW = 2.5, 3.0

# --- regioes de ajuste LOCAL do Setor 1 (centradas na efemeride, +-JAN_H) ---
# formato: [epoca, janela_horas, sigma_upper, sigma_lower, iteracoes]
regioes_local_setor1 = [
    [0,  4, 2.5, 2.0, 12],   # transito 1: re-ajustado para a janela de 6 h
    [2, 13, 1.4, 0.7, 16],   # transito 2: como estava
]


# ------------------------------------------------------------
# ajuste local (do codigo original) - segue a modulacao suave
# em torno do transito, ignorando o transito via sigma_lower
# ------------------------------------------------------------
# DEPOIS
def ajustar_trecho_especifico(t, f, mask_good_flares, t_inicio, t_fim, cadencia_s=120,
                              janela_horas=5, sigma_upper=2.0, sigma_lower=2.0, iteracoes=6,
                              Tc=None, dur_dias=DUR, fator=FATOR):
    buffer_dias = (janela_horas * 1.5) / 24.0
    idx_calc = np.where((t >= t_inicio - buffer_dias) & (t <= t_fim + buffer_dias))[0]
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    if len(idx_alvo) == 0:
        return idx_alvo, np.array([])

    t_calc, f_calc, mask_calc = t[idx_calc], f[idx_calc], mask_good_flares[idx_calc]
    sigma_pontos = (janela_horas * (3600 / cadencia_s)) / 8
    f_limpo = np.copy(f_calc)

    # exclui explicitamente a regiao do transito (pela efemeride) da baseline
    if Tc is not None:
        fora_transito = np.abs(t_calc - Tc) > 0.5 * dur_dias * fator
    else:
        fora_transito = np.ones(t_calc.size, bool)

    for _ in range(iteracoes):
        temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        clipped = sigma_clip(f_calc - temp_smooth, sigma_lower=sigma_lower, sigma_upper=sigma_upper,
                             maxiters=1, cenfunc='median', stdfunc='mad_std')
        mascara_combinada = (~clipped.mask) & mask_calc & fora_transito   # <-- protege o transito
        t_bons, f_bons = t_calc[mascara_combinada], f_limpo[mascara_combinada]
        if len(t_bons) > 2:
            f_limpo = np.interp(t_calc, t_bons, f_bons)
        else:
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]

    modelo_calc = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
    i0 = np.where(idx_calc == idx_alvo[0])[0][0]
    i1 = np.where(idx_calc == idx_alvo[-1])[0][0] + 1
    return idx_alvo, modelo_calc[i0:i1]


# ------------------------------------------------------------
# ajuste polinomial automatico (Setor 27) - um polinomio por
# transito, transito mascarado pela efemeride
# ------------------------------------------------------------
def ajustar_trecho_polinomial(t, f, mask_good_flares, Tc, grau=GRAU, janela=JANELA,
                              dur=DUR, fator=FATOR, sigma_upper=SIG_UP, sigma_lower=SIG_LOW):
    idx_alvo = np.where((t >= Tc - janela) & (t <= Tc + janela))[0]
    if idx_alvo.size < 20:
        return None

    t_alvo, f_alvo = t[idx_alvo], f[idx_alvo]
    dentro = np.abs(t_alvo - Tc) < 0.5 * dur * fator
    oot = (~dentro) & mask_good_flares[idx_alvo]

    if (oot & (t_alvo < Tc)).sum() < grau + 2 or (oot & (t_alvo > Tc)).sum() < grau + 2:
        return None

    clipped = sigma_clip(f_alvo[oot], sigma_lower=sigma_lower, sigma_upper=sigma_upper,
                         maxiters=3, cenfunc='median', stdfunc='mad_std')
    usados = ~clipped.mask
    if usados.sum() < grau + 3:
        return None

    t_centro = np.mean(t_alvo)
    coefs = np.polyfit(t_alvo[oot][usados] - t_centro, f_alvo[oot][usados], deg=grau)
    modelo_poly = np.polyval(coefs, t_alvo - t_centro)

    return dict(idx=idx_alvo, t=t_alvo, flux=f_alvo, modelo=modelo_poly,
                residuo=f_alvo / modelo_poly, dentro=dentro, oot=oot, usados=usados,
                Tc=Tc, t_centro=t_centro, coefs=coefs, grau=grau, tipo='poly')


trechos = []
residuo_por_setor = {}

# ---------- SETOR 1: ajuste LOCAL ----------
if 1 in t_por_setor:
    t_sec, f_sec, mask_sec = t_por_setor[1], f_por_setor[1], mask_por_setor[1]
    residuo_sec = np.full(t_sec.size, np.nan)

    # DEPOIS
    for epoca, jan_h, sig_up, sig_low, iters in regioes_local_setor1:
        # Tc da EFEMERIDE e janela simetrica de +-JAN_H horas
        Tc = T0_REF + epoca * P_REF
        ini, fim = Tc - JANELA, Tc + JANELA

        if not np.any((t_sec >= ini) & (t_sec <= fim)):
            print(f"  [Setor 1] epoca {epoca} ({ini:.4f}-{fim:.4f}) fora dos dados / gap")
            continue

        idx_alvo, mod_local = ajustar_trecho_especifico(
            t_sec, f_sec, mask_sec, ini, fim, 120, jan_h, sig_up, sig_low, iters, Tc=Tc)
        if len(idx_alvo) == 0:
            continue
        residuo_sec[idx_alvo] = f_sec[idx_alvo] / mod_local

        dentro = np.abs(t_sec[idx_alvo] - Tc) < 0.5 * DUR * FATOR
        trechos.append(dict(idx=idx_alvo, t=t_sec[idx_alvo], flux=f_sec[idx_alvo],
                            modelo=mod_local, residuo=f_sec[idx_alvo] / mod_local,
                            dentro=dentro, oot=(~dentro) & mask_sec[idx_alvo],
                            usados=np.ones(((~dentro) & mask_sec[idx_alvo]).sum(), bool),
                            Tc=Tc, t_centro=np.mean(t_sec[idx_alvo]), grau=None,
                            epoca=epoca, setor=1, tipo='local'))

    residuo_por_setor[1] = residuo_sec

# ---------- SETOR 27: ajuste POLINOMIAL automatico ----------
if 27 in t_por_setor:
    t_sec, f_sec, mask_sec = t_por_setor[27], f_por_setor[27], mask_por_setor[27]
    residuo_sec = np.full(t_sec.size, np.nan)

    n_min = int(np.floor((t_sec.min() - T0_REF) / P_REF))
    n_max = int(np.ceil((t_sec.max() - T0_REF) / P_REF))
    for epoca in range(n_min, n_max + 1):
        Tc = T0_REF + epoca * P_REF
        if not (t_sec.min() < Tc < t_sec.max()):
            continue
        r = ajustar_trecho_polinomial(t_sec, f_sec, mask_sec, Tc)
        if r is None:
            print(f"  [Setor 27] epoca {epoca:>3d}: baseline insuficiente / gap")
            continue
        r['epoca'], r['setor'] = epoca, 27
        residuo_sec[r['idx']] = r['residuo']
        trechos.append(r)

    residuo_por_setor[27] = residuo_sec

trechos.sort(key=lambda d: d['epoca'])

# compatibilidade com o resto do codigo
t_setor1  = t_por_setor.get(1)
t_setor27 = t_por_setor.get(27)
residuo_setor1  = residuo_por_setor.get(1)
residuo_setor27 = residuo_por_setor.get(27)

print(f"\nTc = {T0_REF:.5f} +- {T0_ERR:.5f} BTJD | P = {P_REF:.7f} d | T14 = {DUR*24:.2f} +- {DUR_ERR*24:.2f} h")
print(f"Janela de extracao: (-{JAN_H:.0f},+{JAN_H:.0f}) h em torno de Tc\n")
print(f"Trechos ajustados: {len(trechos)}")
for r in trechos:
    extra = f"grau {r['grau']}" if r['tipo'] == 'poly' else "local"
    print(f"  Epoca {r['epoca']:>3d} | Setor {r['setor']:>2d} | {r['tipo']:>5s} ({extra}) "
          f"| Tc~{r['Tc']:.5f} | {r['t'].size} pts "
          f"| cobertura ({(r['t'].min()-r['Tc'])*24:+.1f},{(r['t'].max()-r['Tc'])*24:+.1f}) h")

In [42]:
# ============================================================
# CURVA DE LUZ COMPLETA + MODELO AJUSTADO (por setor)
# ============================================================
for s in t_por_setor:
    t_sec, f_sec = t_por_setor[s], f_por_setor[s]
    res_sec = residuo_por_setor[s]

    # reconstroi o modelo a partir do residuo (modelo = fluxo / residuo) onde houve ajuste
    modelo = np.full(t_sec.size, np.nan)
    tem_aj = np.isfinite(res_sec)
    modelo[tem_aj] = f_sec[tem_aj] / res_sec[tem_aj]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                                   gridspec_kw={'height_ratios': [2, 1]})

    # --- painel de cima: curva de luz + modelo ---
    ax1.plot(t_sec, f_sec, '.', ms=2, color='0.6', alpha=0.6, label='curva de luz')
    ax1.plot(t_sec[tem_aj], modelo[tem_aj], '.', ms=3, color='tab:orange', label='modelo ajustado')
    # marca o centro de cada trecho ajustado
    for r in trechos:
        if r['setor'] == s:
            ax1.axvline(r['Tc'], color='red', ls='--', lw=0.8, alpha=0.5)
            ax1.text(r['Tc'], ax1.get_ylim()[1], f"ep {r['epoca']}",
                     fontsize=7, color='red', ha='center', va='bottom')
    ax1.set_ylabel('fluxo normalizado')
    ax1.set_title(f'AU Mic - Setor {s} - curva completa com ajuste', fontweight='bold')
    ax1.legend(loc='lower left', fontsize=8)
    ax1.grid(alpha=0.25)

    # --- painel de baixo: residuo (fluxo/modelo) so onde ajustou ---
    ax2.plot(t_sec[tem_aj], res_sec[tem_aj], '.', ms=2, color='tab:blue')
    ax2.axhline(1.0, color='k', lw=0.8)
    ax2.set_ylabel('residuo (fluxo/modelo)')
    ax2.set_xlabel('Tempo (BTJD)')
    ax2.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()

In [43]:
# ============================================================
# VERIFICACAO DA QUALIDADE DO AJUSTE
# ============================================================

def diagnostico(r):
    tt, res = r['t'], r['residuo']
    b = np.zeros(res.size, bool)
    if r['tipo'] == 'poly':
        b[np.where(r['oot'])[0][r['usados']]] = True     # pontos que entraram no polinomio
    else:
        b[:] = r['oot']                                  # local: baseline = fora do transito sem flare
    N = b.sum()
    if N < 5:
        return dict(rms=np.nan, ruido=np.nan, razao=np.nan, acf1_sig=np.nan,
                    prof=np.nan, ok=False, mask_base=b)

    rms   = np.std(res[b], ddof=1)
    ruido = mad_std(np.diff(res[b])) / np.sqrt(2)
    razao = rms / ruido

    x = res[b] - res[b].mean()
    acf1 = float(np.sum(x[1:] * x[:-1]) / np.sum(x**2))
    acf1_sig = acf1 * np.sqrt(N)

    nucleo = np.abs(tt - r['Tc']) < 0.25 * DUR
    prof = (1 - np.median(res[nucleo])) * 1e6 if nucleo.sum() > 3 else np.nan

    ok = (razao < 1.30) and (abs(acf1_sig) < 3) and np.isfinite(prof) and (prof > 0)
    return dict(rms=rms * 1e6, ruido=ruido * 1e6, razao=razao, acf1_sig=acf1_sig,
                prof=prof, ok=ok, mask_base=b)


print(f"{'Epoca':>6} {'Set':>4} {'tipo':>6} {'RMS(ppm)':>9} {'ruido':>8} {'RMS/ruido':>10} "
      f"{'ACF1(sig)':>10} {'prof(ppm)':>10}  status")
diags = []
for r in trechos:
    d = diagnostico(r); diags.append(d)
    print(f"{r['epoca']:>6d} {r['setor']:>4d} {r['tipo']:>6s} {d['rms']:>9.0f} {d['ruido']:>8.0f} "
          f"{d['razao']:>10.2f} {d['acf1_sig']:>10.1f} {d['prof']:>10.0f}  "
          f"{'OK' if d['ok'] else 'REVISAR'}")

n_ok = sum(d['ok'] for d in diags)
print(f"\n{n_ok}/{len(diags)} transitos aprovados.")
print("Criterios: RMS/ruido < 1.30 (sem estrutura residual) | |ACF1| < 3 sigma (residuo branco) | profundidade > 0")

# ---------- diagnostico visual ----------
n = len(trechos)
fig, axes = plt.subplots(n, 2, figsize=(11, 2.3 * n), squeeze=False)
for i, (r, d) in enumerate(zip(trechos, diags)):
    h = (r['t'] - r['Tc']) * 24
    ax = axes[i, 0]
    ax.plot(h[r['oot']], r['flux'][r['oot']], '.', ms=3, color='0.6', label='baseline')
    ax.plot(h[r['dentro']], r['flux'][r['dentro']], '.', ms=3, color='tab:blue', label='transito')
    ax.plot(h, r['modelo'], '-', lw=1.5, color='tab:orange', label='modelo')
    ax.axvspan(-0.5 * DUR * 24 * FATOR, 0.5 * DUR * 24 * FATOR, color='tab:blue', alpha=0.07)
    ax.set_ylabel('fluxo norm.')
    ax.set_title(f"Epoca {r['epoca']} | Setor {r['setor']} ({r['tipo']})", fontweight='bold')
    if i == 0: ax.legend(ncol=3, fontsize=7)

    ax = axes[i, 1]
    ax.plot(h[r['oot']], (r['residuo'][r['oot']] - 1) * 1e3, '.', ms=3, color='0.6')
    ax.plot(h[r['dentro']], (r['residuo'][r['dentro']] - 1) * 1e3, '.', ms=3, color='tab:blue')
    ax.axhline(0, color='k', lw=0.8)
    if np.isfinite(d['rms']):
        for sgn in (+1, -1):
            ax.axhline(sgn * d['rms'] / 1e3, color='tab:red', ls=':', lw=0.8)
    ax.set_ylabel('residuo (ppt)')
    ax.set_title(f"RMS/ruido={d['razao']:.2f} | ACF1={d['acf1_sig']:.1f}sig | "
                 f"prof={d['prof']:.0f} ppm | {'OK' if d['ok'] else 'REVISAR'}",
                 color='tab:green' if d['ok'] else 'tab:red')
for ax in axes[-1]:
    ax.set_xlabel('horas desde Tc')
for ax in axes.ravel():
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

 Epoca  Set   tipo  RMS(ppm)    ruido  RMS/ruido  ACF1(sig)  prof(ppm)  status
     0    1  local       352      293       1.20        2.6       3163  OK
     2    1  local       732      492       1.49        7.8       3487  REVISAR
    84   27   poly       307      289       1.06        1.0       2991  OK
    85   27   poly       243      241       1.01       -1.5       2734  OK
    86   27   poly       462      371       1.25        5.0       4146  REVISAR

3/5 transitos aprovados.
Criterios: RMS/ruido < 1.30 (sem estrutura residual) | |ACF1| < 3 sigma (residuo branco) | profundidade > 0


In [45]:
# ============================================================
# CURVA DE LUZ - todos os transitos sobrepostos (-6 a +6 h de Tc)
# ============================================================
fig, ax = plt.subplots(figsize=(9, 6))
cores = plt.cm.viridis(np.linspace(0, 0.9, len(trechos)))

for r, cor in zip(trechos, cores):
    h = (r['t'] - r['Tc']) * 24                     # horas desde Tc
    ax.plot(h, r['flux'], 'o', ms=3, alpha=0.5, color=cor,
            label=f"Epoca {r['epoca']} | Setor {r['setor']} ({r['tipo']})")

ax.axvline(0, color='red', ls='-.', alpha=0.6, label='Tc de referencia')
ax.set_xlim(-6, 6)
ax.set_xlabel('Tempo desde Tc (horas)')
ax.set_ylabel('Fluxo normalizado')
ax.set_title(f'AU Mic b - {len(trechos)} transitos sobrepostos\n'
             f'T0={T0_REF:.6f}  P={P_REF:.6f} d', fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FIGURA FINAL - um transito por linha, janela (-6,+6) h de Tc
#   esquerda : curva de luz original + baseline ajustada
#   direita  : transito final detrended (fluxo / baseline)
# ============================================================
def binar(h, y, largura_h=1/6, hmin=-JAN_H, hmax=JAN_H):
    """media em caixas de largura_h horas -> centro, mediana e erro da mediana"""
    bordas = np.arange(hmin, hmax + largura_h, largura_h)
    qual = np.digitize(h, bordas) - 1
    hb, yb, eb = [], [], []
    for i in range(len(bordas) - 1):
        m = qual == i
        if m.sum() >= 3:
            hb.append(0.5 * (bordas[i] + bordas[i + 1]))
            yb.append(np.median(y[m]))
            eb.append(np.std(y[m], ddof=1) / np.sqrt(m.sum()))
    return np.array(hb), np.array(yb), np.array(eb)


MEIA_DUR = 0.5 * DUR * 24          # 1.75 h  (T14/2 do artigo)
MEIA_ERR = 0.5 * DUR_ERR * 24      # 0.04 h

n = len(trechos)
fig, axes = plt.subplots(n, 2, figsize=(11, 2.3 * n), squeeze=False, sharex=True)

for i, r in enumerate(trechos):
    h = (r['t'] - r['Tc']) * 24

    # ---------- esquerda: curva original + modelo ----------
    ax = axes[i, 0]
    ax.plot(h[r['oot']], r['flux'][r['oot']], '.', ms=3, color='0.6', label='fora de transito')
    ax.plot(h[r['dentro']], r['flux'][r['dentro']], '.', ms=3, color='tab:blue', label='transito')
    ax.plot(h, r['modelo'], '-', lw=1.5, color='tab:orange', label='baseline ajustada')
    ax.axvspan(-MEIA_DUR, MEIA_DUR, color='tab:blue', alpha=0.07)
    ax.set_ylabel('fluxo norm.')
    ax.set_title(f"Epoca {r['epoca']} | Setor {r['setor']} ({r['tipo']})", fontweight='bold')
    if i == 0:
        ax.legend(ncol=3, fontsize=7, loc='lower left')

    # ---------- direita: detrended final ----------
    ax = axes[i, 1]
    ax.plot(h, (r['residuo'] - 1) * 1e3, '.', ms=2.5, color='0.7', alpha=0.8)
    hb, yb, eb = binar(h, r['residuo'])
    ax.errorbar(hb, (yb - 1) * 1e3, yerr=eb * 1e3, fmt='o', ms=4, lw=0,
                elinewidth=0.9, color='tab:red', label='binado 10 min')
    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(0, color='red', ls='-.', lw=0.8, alpha=0.6)
    ax.axvspan(-MEIA_DUR, MEIA_DUR, color='tab:blue', alpha=0.10)
    for s in (-1, 1):                                    # incerteza da duracao
        ax.axvline(s * MEIA_DUR - MEIA_ERR, color='tab:blue', ls=':', lw=0.7)
        ax.axvline(s * MEIA_DUR + MEIA_ERR, color='tab:blue', ls=':', lw=0.7)
    prof = (1 - np.median(r['residuo'][np.abs(h) < 0.25 * DUR * 24])) * 1e6
    ax.set_ylabel('detrended (ppt)')
    ax.set_title(f"profundidade ~ {prof:.0f} ppm", fontsize=8)
    if i == 0:
        ax.legend(fontsize=7, loc='lower left')

for ax in axes[-1]:
    ax.set_xlabel('Tempo desde Tc (horas)')
for ax in axes.ravel():
    ax.set_xlim(-JAN_H, JAN_H)
    ax.grid(alpha=0.25)

fig.suptitle(f'AU Mic b - transitos individuais  |  Tc = {T0_REF:.5f} $\\pm$ {T0_ERR:.5f} BTJD, '
             f'P = {P_REF:.7f} d, T14 = {DUR*24:.2f} $\\pm$ {DUR_ERR*24:.2f} h',
             fontweight='bold', fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# ============================================================
# TRANSITOS EMPILHADOS - somente o ajuste final do detrend
#   cada transito deslocado verticalmente, janela (-6,+6) h de Tc
# ============================================================
DESLOC = 0.008          # deslocamento vertical entre transitos (fluxo normalizado)

n = len(trechos)
fig, ax = plt.subplots(figsize=(7.5, 2.2 + 1.55 * n))
cores = plt.cm.viridis(np.linspace(0, 0.85, n))

for i, (r, cor) in enumerate(zip(trechos, cores)):
    off = (n - 1 - i) * DESLOC              # epoca mais antiga em cima
    h = (r['t'] - r['Tc']) * 24

    ax.plot(h, r['residuo'] + off, '.', ms=2.5, color='0.75', alpha=0.8, zorder=1)
    hb, yb, eb = binar(h, r['residuo'])
    ax.errorbar(hb, yb + off, yerr=eb, fmt='o', ms=4, lw=0, elinewidth=0.9,
                color=cor, zorder=3)
    ax.axhline(1 + off, color=cor, ls=':', lw=0.7, alpha=0.8, zorder=2)
    ax.text(-JAN_H + 0.15, 1 + off + 0.0026,
            f"Epoca {r['epoca']} | Setor {r['setor']} ({r['tipo']})",
            fontsize=8, color=cor, va='bottom', fontweight='bold')

# marcacoes da efemeride do artigo
ax.axvspan(-MEIA_DUR, MEIA_DUR, color='tab:blue', alpha=0.08, zorder=0,
           label=f'T14 = {DUR*24:.2f} $\\pm$ {DUR_ERR*24:.2f} h')
ax.axvline(0, color='red', ls='-.', lw=1.0, alpha=0.7, zorder=2,
           label=f'Tc (T0 = {T0_REF:.5f} $\\pm$ {T0_ERR:.5f})')
for s in (-1, 1):
    ax.axvline(s * MEIA_DUR - MEIA_ERR, color='tab:blue', ls=':', lw=0.7, zorder=2)
    ax.axvline(s * MEIA_DUR + MEIA_ERR, color='tab:blue', ls=':', lw=0.7, zorder=2)

ax.set_xlim(-JAN_H, JAN_H)
ax.set_ylim(1 - 3 * DESLOC / 2, 1 + (n - 0.35) * DESLOC)
ax.set_xlabel('Tempo desde Tc (horas)')
ax.set_ylabel('Fluxo detrended + deslocamento')
ax.set_title(f'AU Mic b - {n} transitos empilhados (detrended)\n'
             f'P = {P_REF:.7f} d | deslocamento = {DESLOC*1e3:.0f} ppt | pontos binados em 10 min',
             fontweight='bold', fontsize=10)
ax.legend(fontsize=8, loc='lower right', framealpha=0.9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TRANSITOS EMPILHADOS EM FASE - todas as epocas na MESMA linha
# de base (e o que esta em empilhado_aumic_b.txt)
# ============================================================
LARG_BIN_H = 0.002 * 24        # 2.88 min - mesma binagem do empilhado_binado_aumic_b.txt

# ---------- dobra em fase ----------
h_todos, n_todos = [], []
for r in trechos:
    h_todos.append((r['t'] - r['Tc']) * 24)
    n_todos.append(r['residuo'] / np.median(r['residuo'][r['oot']]))   # igual ao .txt

h_all = np.concatenate(h_todos)
n_all = np.concatenate(n_todos)
ordem = np.argsort(h_all)
h_all, n_all = h_all[ordem], n_all[ordem]

hb, yb, eb = binar(h_all, n_all, largura_h=LARG_BIN_H)

# ---------- profundidade combinada ----------
nucleo = np.abs(hb) < 0.25 * DUR * 24                 # +-0.44 h em torno de Tc
prof_comb = (1 - np.median(yb[nucleo])) * 1e6
prof_err  = np.median(eb[nucleo]) / np.sqrt(nucleo.sum()) * 1e6

# ---------- figura ----------
fig, ax = plt.subplots(figsize=(9, 6))
cores = plt.cm.viridis(np.linspace(0, 0.85, len(trechos)))

for r, cor, h, nn in zip(trechos, cores, h_todos, n_todos):
    ax.plot(h, nn, '.', ms=3, alpha=0.35, color=cor,
            label=f"Epoca {r['epoca']} | Setor {r['setor']}", zorder=1)

ax.errorbar(hb, yb, yerr=eb, fmt='o', ms=5, lw=0, elinewidth=1.1, color='tab:red',
            zorder=4, label=f'mediana binada ({LARG_BIN_H*60:.1f} min)')

ax.axhline(1.0, color='k', lw=0.8, zorder=2)
ax.axvline(0, color='red', ls='-.', lw=1.0, alpha=0.7, zorder=2,
           label=f'Tc (T0 = {T0_REF:.5f} $\\pm$ {T0_ERR:.5f})')
ax.axvspan(-MEIA_DUR, MEIA_DUR, color='tab:blue', alpha=0.08, zorder=0,
           label=f'T14 = {DUR*24:.2f} $\\pm$ {DUR_ERR*24:.2f} h')
for s in (-1, 1):
    ax.axvline(s * MEIA_DUR - MEIA_ERR, color='tab:blue', ls=':', lw=0.7, zorder=2)
    ax.axvline(s * MEIA_DUR + MEIA_ERR, color='tab:blue', ls=':', lw=0.7, zorder=2)

# folga no topo para a legenda nao cobrir os pontos
ylo, yhi = ax.get_ylim()
ax.set_ylim(ylo, yhi + 0.32 * (yhi - ylo))

ax.annotate(f'profundidade combinada\n{prof_comb:.0f} $\\pm$ {prof_err:.0f} ppm',
            xy=(0, np.median(yb[nucleo])), xytext=(0.04, 0.09), textcoords='axes fraction',
            fontsize=9, fontweight='bold', color='tab:red', ha='left',
            arrowprops=dict(arrowstyle='->', color='tab:red', lw=0.9))

ax.set_xlim(-JAN_H, JAN_H)
ax.set_xlabel('Tempo desde Tc (horas)')
ax.set_ylabel('Fluxo detrended normalizado')
ax.set_title(f'AU Mic b - {len(trechos)} transitos empilhados em fase '
             f'({h_all.size} pontos)\nT0 = {T0_REF:.6f}  P = {P_REF:.7f} d',
             fontweight='bold', fontsize=10)
ax.legend(fontsize=8, ncol=2, loc='upper right', framealpha=0.9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f"Empilhado em fase: {h_all.size} pontos | {len(hb)} bins de {LARG_BIN_H*60:.1f} min")
print(f"Profundidade combinada (|dt| < {0.25*DUR*24:.2f} h): {prof_comb:.0f} +- {prof_err:.0f} ppm")

In [ ]:
# ============================================================
# SALVA OS TRANSITOS DETRENDED EM .txt (pasta transitos_aumic_b)
#   mesmo formato dos arquivos ja existentes
#   somente ate a epoca EPOCA_MAX (setores 1 e 27)
# ============================================================
PASTA      = "transitos_aumic_b"
EPOCA_MAX  = 86
ANO_SETOR  = {1: 2018, 27: 2020}
BIN_DIAS   = 0.002

os.makedirs(PASTA, exist_ok=True)

salvar = [r for r in trechos if r['epoca'] <= EPOCA_MAX]
setores_txt = " e ".join(str(s) for s in sorted({r['setor'] for r in salvar}))

lin_todos, lin_empilha, resumo = [], [], []

for r in salvar:
    ano  = ANO_SETOR[r['setor']]
    cod  = int(f"{r['setor']}{ano}")          # 12018, 272020, ...
    dt   = r['t'] - r['Tc']                   # dias desde Tc
    res  = r['residuo']                       # fluxo / baseline ajustada
    # renormaliza pela mediana fora do transito (sem flares)
    norm = res / np.median(res[r['oot']])

    arq = os.path.join(PASTA, f"transito_ep{r['epoca']:03d}_setor{cod}.txt")
    with open(arq, 'w') as fh:
        fh.write(f"# AU Mic b | epoca={r['epoca']} | Setor {r['setor']:<2d} ({ano}) | "
                 f"Tc={r['Tc']:.6f} | T0={T0_REF:.6f} | P={P_REF}\n")
        fh.write("# tempo_BTJD  dt_dias  fluxo_residual  fluxo_normalizado\n")
        for tt, dd, rr, nn in zip(r['t'], dt, res, norm):
            fh.write(f"{tt:.8f} {dd:.8f} {rr:.8f} {nn:.8f}\n")
    print(f"  {arq}  ({r['t'].size} pts)")

    for tt, dd, rr, nn in zip(r['t'], dt, res, norm):
        lin_todos.append(f"{tt:.8f} {dd:.8f} {rr:.8f} {nn:.8f} {r['epoca']} {cod}\n")
    for dd, nn in zip(dt, norm):
        lin_empilha.append((dd, nn, r['epoca']))

    resumo.append((r['epoca'], cod, r['Tc'], r['t'].min(), r['t'].max(), r['t'].size))

# ---------- todos os transitos num arquivo so ----------
arq = os.path.join(PASTA, "todos_transitos_aumic_b.txt")
with open(arq, 'w') as fh:
    fh.write(f"# AU Mic b - todos os transitos extraidos dos setores {setores_txt}\n")
    fh.write(f"# T0_ref={T0_REF:.6f}  P_ref={P_REF}  janela=+/-{JANELA:.2f} d  "
             f"N_transitos={len(salvar)}  N_pontos={len(lin_todos)}\n")
    fh.write("# tempo_BTJD  dt_dias  fluxo_residual  fluxo_normalizado  epoca  setor\n")
    fh.writelines(lin_todos)
print(f"  {arq}  ({len(lin_todos)} pts)")

# ---------- empilhamento em fase ----------
lin_empilha.sort(key=lambda z: z[0])
dt_emp   = np.array([z[0] for z in lin_empilha])
flx_emp  = np.array([z[1] for z in lin_empilha])
ep_emp   = np.array([z[2] for z in lin_empilha])

arq = os.path.join(PASTA, "empilhado_aumic_b.txt")
with open(arq, 'w') as fh:
    fh.write(f"# AU Mic b - empilhamento em fase ({len(salvar)} transitos)\n")
    fh.write(f"# T0_ref={T0_REF:.6f}  P_ref={P_REF}\n")
    fh.write("# dt_dias  fluxo_normalizado  epoca\n")
    for dd, nn, ee in zip(dt_emp, flx_emp, ep_emp):
        fh.write(f"{dd:.8f} {nn:.8f} {ee}\n")
print(f"  {arq}  ({dt_emp.size} pts)")

# ---------- empilhado binado ----------
bordas = np.arange(-JANELA, JANELA + BIN_DIAS, BIN_DIAS)
qual   = np.digitize(dt_emp, bordas) - 1
arq = os.path.join(PASTA, "empilhado_binado_aumic_b.txt")
n_bins = 0
with open(arq, 'w') as fh:
    fh.write(f"# AU Mic b - empilhado binado (bins de {BIN_DIAS:.3f} d)\n")
    fh.write(f"# T0_ref={T0_REF:.6f}  P_ref={P_REF}\n")
    fh.write("# dt_dias  fluxo_normalizado  erro\n")
    for i in range(len(bordas) - 1):
        m = qual == i
        if m.sum() < 3:
            continue
        fh.write(f"{np.mean(dt_emp[m]):.8f} {np.median(flx_emp[m]):.8f} "
                 f"{np.std(flx_emp[m], ddof=1)/np.sqrt(m.sum()):.8f}\n")
        n_bins += 1
print(f"  {arq}  ({n_bins} bins)")

# ---------- resumo ----------
arq = os.path.join(PASTA, "resumo_transitos.txt")
with open(arq, 'w') as fh:
    fh.write(f"# AU Mic b | T0_ref={T0_REF:.6f} | P_ref={P_REF} | janela=+/-{JANELA:.2f} d\n")
    fh.write("#  epoca  setor        Tc_BTJD          t_ini          t_fim  N_pts\n")
    for ep, cod, tc, ti, tf, n_pts in resumo:
        fh.write(f"{ep:8d} {cod:6d}{tc:15.6f}{ti:15.6f}{tf:15.6f}{n_pts:7d}\n")
print(f"  {arq}")

print(f"\n{len(salvar)} transitos salvos em {PASTA}/ (epocas <= {EPOCA_MAX}, setores {setores_txt})")